# Phase B Probe Evaluation - rebuilt graph (H10 values-as-properties, H12 passages)

**Author**: Knowledge Graph Foundry autonomous build

Runs the 28-probe set against the H10 rebuild (second Neo4j: 19 cured types, values as properties, Chunk provenance nodes, propositions + similarity edges from optimize). B0 = full stack. B1 = H12 ablation: PPR projection restricted to Entity nodes (chunks excluded) by monkeypatching ppr_query, isolating the passage-node effect on the same graph.

In [ ]:
# imports
import datetime
import json
import re
from pathlib import Path

import yaml

from knowledge_graph_foundry import Foundry, load_settings

In [ ]:
# configuration - the rebuilt graph on the second Neo4j
URI = 'bolt://user-konrad.jelen-kgf-neo4j2:7687'
probes = yaml.safe_load(Path('../tests/probes/cpap-probe-set.yml').read_text())

def base_settings():
    s = load_settings(Path('../config.yml') if Path('../config.yml').exists() else None)
    s.neo4j.uri = URI
    s.neo4j.user = 'neo4j'
    s.neo4j.password = 'kgfoundry'
    s.graphrag.propositions_enabled = True
    s.graphrag.abstention_enabled = False
    return s
print(len(probes), 'probes against', URI)

In [ ]:
# scoring (same deterministic rules as probe_eval.ipynb)
REFUSAL = re.compile(
    r'no information|not (?:available|stated|specified|mentioned|provided)|'
    r'lacks|does not (?:contain|include|specify|state|provide|mention)|'
    r'cannot answer|unable to|unanswerable|no (?:data|details|answer)|'
    r'does not contain enough information', re.I)

def _norm(s):
    return re.sub(r'\s+', ' ', s.casefold())

def evidence_recall(gold, context):
    ctx = _norm(context)
    return sum(1 for g in gold if _norm(g) in ctx) / len(gold) if gold else None

def value_tokens(gold_answer):
    return re.findall(r'[\w.\-/]*\d[\w.\-/]*', gold_answer)

def answer_correct(probe, answer):
    ans = _norm(answer)
    if probe['category'] == 'unanswerable':
        return bool(REFUSAL.search(answer))
    tokens = value_tokens(probe['gold_answer'])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    return _norm(probe['gold_answer']) in ans

def run_eval(foundry, label):
    rows = []
    for p in probes:
        lines, _, _ = foundry._retrieve_local(p['question'])
        context = '\n'.join(lines)
        answer = foundry.query(p['question'])['answer']
        rows.append({'id': p['id'], 'category': p['category'],
                     'evidence_recall': evidence_recall(p['gold_evidence'], context),
                     'correct': answer_correct(p, answer),
                     'context_chars': len(context), 'answer': answer})
        print(p['id'], p['category'], rows[-1]['evidence_recall'], rows[-1]['correct'])
    return {'label': label, 'rows': rows}

def summarize(result):
    rows = result['rows']
    answerable = [r for r in rows if r['category'] != 'unanswerable']
    unans = [r for r in rows if r['category'] == 'unanswerable']
    recalls = [r['evidence_recall'] for r in answerable if r['evidence_recall'] is not None]
    by_cat = {c: (lambda rs: sum(r['correct'] for r in rs) / len(rs) if rs else None)(
        [r for r in rows if r['category'] == c])
        for c in ('single_fact', 'comparison', 'multi_hop')}
    return {'label': result['label'],
            'evidence_recall': sum(recalls) / len(recalls) if recalls else 0,
            'answer_accuracy': sum(r['correct'] for r in answerable) / len(answerable),
            'accuracy_by_category': by_cat,
            'correct_refusal': sum(r['correct'] for r in unans) / len(unans),
            'avg_context_chars': int(sum(r['context_chars'] for r in rows) / len(rows))}

In [ ]:
# B0: full stack on the rebuilt graph
results = {}
with Foundry(base_settings()) as f:
    results['B0_full'] = run_eval(f, 'B0_full')
print(summarize(results['B0_full']))

In [ ]:
# B1: H12 ablation - chunks OUT of the PPR projection (same graph)
import knowledge_graph_foundry.graph.graphrag as gr

gr.PPR_NODE_LABELS = ('Entity',)  # designed ablation seam
with Foundry(base_settings()) as f:
    results['B1_no_passages'] = run_eval(f, 'B1_no_passages')
gr.PPR_NODE_LABELS = ('Entity', 'Chunk')  # restore
print(summarize(results['B1_no_passages']))

In [ ]:
# summary + persist
summaries = [summarize(r) for r in results.values()]
for s in summaries:
    print(f"{s['label']:<16} evrecall={s['evidence_recall']:.3f} "
          f"acc={s['answer_accuracy']:.3f} by_cat={s['accuracy_by_category']} "
          f"refusal_ok={s['correct_refusal']} ctx={s['avg_context_chars']}")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
out = Path('../reports') / f'probe-eval-phaseB-{stamp}.json'
out.write_text(json.dumps({'summaries': summaries, 'results': results}, indent=2, default=str))
print('saved', out)